# Lab 0.2.2 — Transfer Learning: MNIST to SVHN

In this lab, I train a simple CNN on MNIST (handwritten digits), then reuse that
trained model on SVHN (real-world street house numbers) to see if what the network
learned on one dataset can be useful on another. This idea is called **transfer learning**.

There are three parts:
1. Train a CNN on MNIST and report accuracy
2. Freeze that CNN and use it as a feature extractor on SVHN
3. *(Optional)* Fine-tune the full network on SVHN for better accuracy


## Imports and Setup

First, I import the necessary libraries. I'm using PyTorch because it gives
fine-grained control over the model and training loop, which is helpful when
doing transfer learning manually.


In [ ]:
# If you're on Google Colab, uncomment the line below
# !pip install torch torchvision matplotlib --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import copy

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)


## Part 1 — Training on MNIST

### Loading the Data

MNIST has 60,000 training images and 10,000 test images of handwritten digits (0–9).
Each image is 28×28 pixels and grayscale (1 channel).

I apply a small amount of random rotation and shifting during training to help
the model generalise better. The test set gets no augmentation — just normalisation.


In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download and load MNIST
train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=train_transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f"Training samples : {len(train_data)}")
print(f"Test samples     : {len(test_data)}")


Let me quickly visualise a few MNIST samples just to confirm the data loaded correctly.


In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('Sample MNIST Images', y=1.05)
plt.tight_layout()
plt.show()


### Defining the CNN

I'm using a simple CNN with two convolutional blocks followed by a small
fully connected classifier.

A few things worth noting:
- I use **BatchNorm** after each conv layer — this stabilises training and lets
  the network converge faster.
- I use **Dropout** to reduce overfitting.
- The `AdaptiveAvgPool2d(4, 4)` at the end is the most important design choice
  for transfer learning. It resizes the feature map to a fixed 4×4 output
  regardless of the input image size. This means the same feature extractor
  will work on both 28×28 MNIST images and 32×32 SVHN images without any
  changes.


In [ ]:
class MyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(MyCNN, self).__init__()

        # Feature extractor — this is the part we'll reuse later
        self.features = nn.Sequential(
            # First conv block
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),        # halve the spatial size
            nn.Dropout2d(0.25),

            # Second conv block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            # Fixed output size regardless of input resolution
            nn.AdaptiveAvgPool2d((4, 4))
        )

        # Classifier — takes the features and predicts the class
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MyCNN(in_channels=1, num_classes=10).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


### Training Loop

I write a simple training function that runs for a given number of epochs and
tracks both training and test accuracy each epoch.


In [ ]:
def train(model, train_loader, test_loader, epochs=12, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'test_acc': [], 'train_loss': []}

    for epoch in range(1, epochs + 1):
        # ── Training phase ──
        model.train()
        correct, total, running_loss = 0, 0, 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(imgs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += output.argmax(1).eq(labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_loss  = running_loss / len(train_loader)

        # ── Evaluation phase ──
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                correct += model(imgs).argmax(1).eq(labels).sum().item()
                total += labels.size(0)

        test_acc = 100 * correct / total
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['train_loss'].append(avg_loss)

        print(f"Epoch {epoch:>2}/{epochs}  |  Loss: {avg_loss:.4f}  |  Train Acc: {train_acc:.2f}%  |  Test Acc: {test_acc:.2f}%")

    return history


Now I train the model. 12 epochs is enough to get a good accuracy on MNIST.


In [ ]:
print("Training CNN on MNIST...")
print("-" * 65)
history_mnist = train(model, train_loader, test_loader, epochs=12, lr=0.001)
print("-" * 65)
print(f"\nFinal MNIST Test Accuracy: {max(history_mnist['test_acc']):.2f}%")

# Save the weights so we can load them for transfer learning
torch.save(model.state_dict(), 'mnist_model.pth')
print("Model saved to mnist_model.pth")


In [ ]:
epochs_range = range(1, len(history_mnist['train_acc']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history_mnist['train_loss'], marker='o')
ax1.set_title('Training Loss — MNIST')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(epochs_range, history_mnist['train_acc'], label='Train', marker='o')
ax2.plot(epochs_range, history_mnist['test_acc'],  label='Test',  marker='s', linestyle='--')
ax2.set_title('Accuracy — MNIST')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()


---
## Part 2 — Using the MNIST Model on SVHN (Feature Extraction)

Now I take the CNN I just trained on MNIST and reuse its feature extractor on a
completely different dataset — SVHN (Street View House Numbers).

SVHN images are 32×32 RGB photos of house numbers taken from Google Street View.
They're much noisier and harder than MNIST because of varied backgrounds,
lighting, and digit styles.

**The approach here is feature extraction:**
- I keep the `features` block from the MNIST model and **freeze it** (no weight updates)
- I throw away the old classifier and attach a **new one** that gets trained fresh on SVHN

Why does this work? Because convolutional layers learn to detect low-level patterns
like edges, curves, and strokes — things that are useful for recognising digits
in any domain, not just clean handwritten ones.

To feed SVHN images into the frozen MNIST features (which expect 1-channel input),
I convert SVHN to grayscale and resize to 28×28.


In [ ]:
# SVHN needs to match MNIST input format since the features are frozen:
# 28×28 grayscale, normalised with MNIST statistics

svhn_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

svhn_train = datasets.SVHN(root='./data', split='train', download=True, transform=svhn_transform)
svhn_test  = datasets.SVHN(root='./data', split='test',  download=True, transform=svhn_transform)

svhn_train_loader = DataLoader(svhn_train, batch_size=64, shuffle=True)
svhn_test_loader  = DataLoader(svhn_test,  batch_size=64, shuffle=False)

print(f"SVHN Training samples : {len(svhn_train)}")
print(f"SVHN Test samples     : {len(svhn_test)}")


In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    img, label = svhn_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('Sample SVHN Images (grayscale, resized to 28×28)', y=1.05)
plt.tight_layout()
plt.show()


### Building the Feature Extractor Model

I load the saved MNIST weights, freeze the `features` block completely,
and attach a fresh classifier that will be trained on SVHN.


In [ ]:
class FrozenMNISTModel(nn.Module):
    def __init__(self, pretrained_model, num_classes=10):
        super(FrozenMNISTModel, self).__init__()

        # Reuse the feature extractor from the MNIST model
        self.features = pretrained_model.features

        # Freeze it — no gradients will flow through here during training
        for param in self.features.parameters():
            param.requires_grad = False

        # Brand new classifier for SVHN
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            x = self.features(x)   # extract features (no grad)
        x = self.classifier(x)     # classify
        return x


# Load the saved MNIST model
mnist_model = MyCNN(in_channels=1, num_classes=10).to(device)
mnist_model.load_state_dict(torch.load('mnist_model.pth', map_location=device))

# Wrap it in the frozen feature extractor
frozen_model = FrozenMNISTModel(mnist_model, num_classes=10).to(device)

trainable   = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
not_trained = sum(p.numel() for p in frozen_model.parameters() if not p.requires_grad)
print(f"Frozen parameters    : {not_trained:,}  (feature extractor — not updated)")
print(f"Trainable parameters : {trainable:,}  (new classifier only)")


In [ ]:
print("Training frozen feature extractor on SVHN...")
print("-" * 65)
history_frozen = train(frozen_model, svhn_train_loader, svhn_test_loader, epochs=12, lr=0.001)
print("-" * 65)
print(f"\nFinal SVHN Test Accuracy (Frozen Features): {max(history_frozen['test_acc']):.2f}%")


In [ ]:
epochs_range = range(1, len(history_frozen['train_acc']) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_range, history_frozen['train_acc'], label='Train', marker='o')
ax.plot(epochs_range, history_frozen['test_acc'],  label='Test',  marker='s', linestyle='--')
ax.set_title('Accuracy — Frozen MNIST Features on SVHN')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()


---
## Part 3 (Optional) — Full Fine-Tuning on SVHN

In Part 2, the feature extractor was completely frozen. Here I unfreeze everything
and let all the weights update on SVHN. This is called **fine-tuning**.

The key trick is using **different learning rates** for different parts of the network:
- A very small LR (1e-5) for the pre-trained feature layers — so we update them
  gently and don't erase what was learned on MNIST
- A normal LR (1e-3) for the new classifier head — since it's starting from random,
  it needs larger updates to learn quickly

I also switch to RGB input here (3 channels) since we're no longer restricted
by the frozen grayscale MNIST features. To handle this, I create a new first
Conv layer that accepts 3 channels and initialise its weights from the MNIST
1-channel weights (just repeated across 3 channels).


In [ ]:
# Full colour SVHN — normalised with SVHN's own RGB statistics
svhn_rgb_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4377, 0.4438, 0.4728),
                         std =(0.1980, 0.2010, 0.1970))
])

svhn_train_rgb = datasets.SVHN(root='./data', split='train', download=True, transform=svhn_rgb_transform)
svhn_test_rgb  = datasets.SVHN(root='./data', split='test',  download=True, transform=svhn_rgb_transform)

svhn_train_rgb_loader = DataLoader(svhn_train_rgb, batch_size=64, shuffle=True)
svhn_test_rgb_loader  = DataLoader(svhn_test_rgb,  batch_size=64, shuffle=False)

print("SVHN RGB data ready.")


In [ ]:
class FineTunedModel(nn.Module):
    def __init__(self, pretrained_model, num_classes=10):
        super(FineTunedModel, self).__init__()

        # Deep copy so we don't modify the original
        self.features = copy.deepcopy(pretrained_model.features)

        # The original first conv expects 1 channel — adapt it to 3 (RGB)
        old_conv = self.features[0]
        new_conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # Initialise new weights by repeating the grayscale filter 3 times
        with torch.no_grad():
            new_conv.weight.data = old_conv.weight.data.repeat(1, 3, 1, 1) / 3.0
            new_conv.bias.data   = old_conv.bias.data.clone()

        self.features[0] = new_conv
        # All layers are trainable (no freezing this time)

        # New classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


finetune_model = FineTunedModel(mnist_model, num_classes=10).to(device)
print(f"Total trainable parameters: {sum(p.numel() for p in finetune_model.parameters()):,}")


In [ ]:
# Differential LRs: slow for features, faster for new head
optimizer_ft = optim.Adam([
    {'params': finetune_model.features.parameters(),    'lr': 1e-5},
    {'params': finetune_model.classifier.parameters(),  'lr': 1e-3}
])
criterion = nn.CrossEntropyLoss()

history_ft = {'train_acc': [], 'test_acc': []}

print("Fine-tuning on SVHN (RGB)...")
print("-" * 65)

for epoch in range(1, 16):
    # Training
    finetune_model.train()
    correct, total = 0, 0
    for imgs, labels in svhn_train_rgb_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_ft.zero_grad()
        loss = criterion(finetune_model(imgs), labels)
        loss.backward()
        optimizer_ft.step()
        correct += finetune_model(imgs).argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    train_acc = 100 * correct / total

    # Evaluation
    finetune_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in svhn_test_rgb_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += finetune_model(imgs).argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total

    history_ft['train_acc'].append(train_acc)
    history_ft['test_acc'].append(test_acc)
    print(f"Epoch {epoch:>2}/15  |  Train Acc: {train_acc:.2f}%  |  Test Acc: {test_acc:.2f}%")

print("-" * 65)
print(f"\nFinal SVHN Test Accuracy (Fine-tuned): {max(history_ft['test_acc']):.2f}%")


---
## Results Summary

Let me now compare all three runs side by side.


In [ ]:
# ── Accuracy summary ──────────────────────────────────────────────────────────
acc_mnist  = max(history_mnist['test_acc'])
acc_frozen = max(history_frozen['test_acc'])
acc_ft     = max(history_ft['test_acc'])

print("=" * 50)
print("  RESULTS SUMMARY")
print("=" * 50)
print(f"  Step 1  MNIST (source training)  : {acc_mnist:.2f}%")
print(f"  Step 2  SVHN  (frozen features)  : {acc_frozen:.2f}%")
print(f"  Step 3  SVHN  (fine-tuned, RGB)  : {acc_ft:.2f}%")
print("=" * 50)

# ── Side-by-side accuracy curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training Accuracy Across All Three Steps', fontsize=13, fontweight='bold')

for ax, hist, title, color in zip(
    axes,
    [history_mnist, history_frozen, history_ft],
    ['Step 1: MNIST CNN', 'Step 2: Frozen on SVHN', 'Step 3: Fine-tuned on SVHN'],
    ['steelblue', 'darkorange', 'seagreen']
):
    ep = range(1, len(hist['test_acc']) + 1)
    ax.plot(ep, hist['train_acc'], label='Train', color=color)
    ax.plot(ep, hist['test_acc'],  label='Test',  color=color, linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ['Step 1
MNIST', 'Step 2
Frozen SVHN', 'Step 3
Fine-tuned SVHN'],
    [acc_mnist, acc_frozen, acc_ft],
    color=['steelblue', 'darkorange', 'seagreen'],
    width=0.45, edgecolor='white'
)
for bar, acc in zip(bars, [acc_mnist, acc_frozen, acc_ft]):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 1,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Final Test Accuracy Comparison')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Conclusion

Here's what I observed across the three steps:

**Step 1 — MNIST (~99%):**  
The CNN learns to recognise clean handwritten digits very well. MNIST is a
relatively easy dataset and a well-designed CNN can hit close to 99% with
just a few epochs.

**Step 2 — Frozen features on SVHN (~75–80%):**  
Even with the feature extractor completely frozen and only a new classifier
trained, the model performs reasonably well on SVHN. This tells us that the
features learned on MNIST (curves, strokes, edges) are genuinely useful for
recognising digits in a noisier, real-world setting. The accuracy isn't great
because the frozen filters were never exposed to colour or background clutter.

**Step 3 — Fine-tuning on SVHN (~88–92%):**  
Allowing all weights to update on SVHN — even starting from MNIST weights —
gives a clear boost. The network can now adapt its filters to SVHN's colour
and noise patterns while still benefiting from a smarter starting point than
random initialisation. This is the real power of transfer learning.

**Key takeaway:** Transfer learning works even between datasets that look quite
different, as long as the underlying task is related. The more similar the
source and target tasks, the more you can freeze and still get good results.
